# Citus Database Setup and SDK Demo (Single Node)

This notebook demonstrates how to set up a Citus database using a single-node docker container, populate the patch table with dummy data, and implement a Python SDK for interacting with the database. Worker node logic is omitted for single-node setup.

## 1. Install and Import Required Libraries

Install `psycopg` if not already installed, and import all required libraries for database interaction.

In [1]:
# Install psycopg if needed (uncomment if running in a new environment)
# !pip install psycopg[binary]

import psycopg
from psycopg.rows import dict_row
import random
import datetime
import base64
from db_client import CitusHeadClient
import os

In [2]:
# Import DB connection constants from constants.py
from constants import (
    CITUS_HEAD_HOST, CITUS_HEAD_PORT, CITUS_HEAD_DB, CITUS_HEAD_USER, CITUS_HEAD_PASSWORD
)

In [3]:
# Set DB connection variables from constants (single-node)
DB_HOST = CITUS_HEAD_HOST
DB_PORT = CITUS_HEAD_PORT
DB_NAME = CITUS_HEAD_DB
DB_USER = CITUS_HEAD_USER
DB_PASSWORD = CITUS_HEAD_PASSWORD

In [4]:
NUM_PATCHES = 100

## 0. Drop All Tables (Clean Start)

Drop all tables if they exist to ensure a clean setup.

In [5]:
head_client = CitusHeadClient(DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD)
head_client.drop_all_tables()
print("All tables dropped (if existed).")


All tables dropped (if existed).


## 2. Connect to Citus Node

Establish a connection to the Citus/Postgres node using psycopg. Store connection parameters securely (e.g., using environment variables).

In [6]:
# Use CitusHeadClient for connection
def get_head_connection():
    return CitusHeadClient(DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD).get_connection()

# Test connection
with get_head_connection() as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT version();')
        print('Connected to:', cur.fetchone()['version'])

Connected to: PostgreSQL 18.1 (Debian 18.1-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## 3. Create Database Schema (Tables)

Create all tables as described in the technical design document, including distributed and reference tables. Use Citus distribution commands where required.

In [7]:
head_client.setup_schema()
print("Schema and distribution setup complete.")


Schema and distribution setup complete.


In [8]:
head_client.setup_triggers()


INSERT trigger function created on coordinator and workers.
UPDATE trigger function created on coordinator and workers.
Per-shard INSERT triggers installed on pred_patch_latest shards.
Per-shard UPDATE triggers installed on patch shards.

All per-shard triggers installed.


## 4. Verify Table Creation

Query the information schema to verify that all tables have been created successfully.

In [9]:
# List all tables in the public schema
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_name FROM information_schema.tables
            WHERE table_schema = 'public'
            ORDER BY table_name;
        """)
        tables = [row['table_name'] for row in cur.fetchall()]
        print('Tables in public schema:', tables)

Tables in public schema: ['citus_schemas', 'citus_tables', 'confusion_matrix_ln', 'image', 'label_class', 'patch', 'pred_patch_last', 'pred_patch_latest', 'project', 'settings']


## 5. Insert Dummy Data into Patch Table

Generate and insert dummy data into the patch table, ensuring all required fields are populated and constraints are respected.

In [10]:
# Helper: Insert dummy project, image, label_class for FK constraints
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("INSERT INTO project (project_name, description) VALUES (%s, %s) RETURNING project_id;", ('Demo Project', 'For dummy data'))
        project_id = cur.fetchone()['project_id']
        cur.execute("INSERT INTO image (project_id, name, image_path, upload_ts, base_mag, base_width, base_height, deepzoom_tilesize) VALUES (%s, %s, %s, %s, %s, %s, %s, %s) RETURNING image_id;",
                    (project_id, 'Demo Image', '/tmp/demo.tif', datetime.datetime.now(), 20.0, 10000, 8000, 256))
        image_id = cur.fetchone()['image_id']
        cur.execute("INSERT INTO label_class (project_id, name, color_code, event_ts) VALUES (%s, %s, %s, %s) RETURNING label_class_id;",
                    (project_id, 'Tumor', '#FF0000', datetime.datetime.now()))
        label_class_id = cur.fetchone()['label_class_id']
        print(f"Inserted project_id={project_id}, image_id={image_id}, label_class_id={label_class_id}")

def random_bytes(size=128):
    return os.urandom(size)

for i in range(NUM_PATCHES):
    patch_id = head_client.insert_patch(1000 + i, label_class_id, image_id, 20.0, random_bytes())
    print(f"Inserted patch_id={patch_id}")

Inserted project_id=1, image_id=1, label_class_id=1
Inserted patch_id=1
Inserted patch_id=2
Inserted patch_id=3
Inserted patch_id=4
Inserted patch_id=5
Inserted patch_id=6
Inserted patch_id=7
Inserted patch_id=8
Inserted patch_id=9
Inserted patch_id=10
Inserted patch_id=11
Inserted patch_id=12
Inserted patch_id=13
Inserted patch_id=14
Inserted patch_id=15
Inserted patch_id=16
Inserted patch_id=17
Inserted patch_id=18
Inserted patch_id=19
Inserted patch_id=20
Inserted patch_id=21
Inserted patch_id=22
Inserted patch_id=23
Inserted patch_id=24
Inserted patch_id=25
Inserted patch_id=26
Inserted patch_id=27
Inserted patch_id=28
Inserted patch_id=29
Inserted patch_id=30
Inserted patch_id=31
Inserted patch_id=32
Inserted patch_id=33
Inserted patch_id=34
Inserted patch_id=35
Inserted patch_id=36
Inserted patch_id=37
Inserted patch_id=38
Inserted patch_id=39
Inserted patch_id=40
Inserted patch_id=41
Inserted patch_id=42
Inserted patch_id=43
Inserted patch_id=44
Inserted patch_id=45
Inserted pat

In [11]:
# Check number of shards for the patch table and print row counts per shard, including empty shards
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        # Number of shards
        cur.execute("SELECT count(*) FROM pg_dist_shard WHERE logicalrelid = 'patch'::regclass;")
        num_shards = cur.fetchone()['count']
        print(f"Number of shards for 'patch' table: {num_shards}")
        # Row counts per shard, including empty
        cur.execute("""
            SELECT s.shardid, COALESCE(count(p.patch_id), 0) as row_count
            FROM pg_dist_shard s
            LEFT JOIN patch p ON get_shard_id_for_distribution_column('patch', p.patch_id) = s.shardid
            WHERE s.logicalrelid = 'patch'::regclass
            GROUP BY s.shardid
            ORDER BY s.shardid;
        """)
        rows = cur.fetchall()
        empty_count = 0
        for row in rows:
            print(f"Shard {row['shardid']}: {row['row_count']} rows")
            if row['row_count'] == 0:
                empty_count += 1
        print(f"Empty shards: {empty_count} out of {num_shards}")

Number of shards for 'patch' table: 32
Shard 110700: 3 rows
Shard 110701: 5 rows
Shard 110702: 2 rows
Shard 110703: 6 rows
Shard 110704: 3 rows
Shard 110705: 1 rows
Shard 110706: 2 rows
Shard 110707: 2 rows
Shard 110708: 6 rows
Shard 110709: 2 rows
Shard 110710: 2 rows
Shard 110711: 2 rows
Shard 110712: 3 rows
Shard 110713: 3 rows
Shard 110714: 5 rows
Shard 110715: 2 rows
Shard 110716: 3 rows
Shard 110717: 2 rows
Shard 110718: 0 rows
Shard 110719: 3 rows
Shard 110720: 6 rows
Shard 110721: 3 rows
Shard 110722: 3 rows
Shard 110723: 3 rows
Shard 110724: 3 rows
Shard 110725: 2 rows
Shard 110726: 1 rows
Shard 110727: 2 rows
Shard 110728: 7 rows
Shard 110729: 3 rows
Shard 110730: 5 rows
Shard 110731: 5 rows
Empty shards: 1 out of 32


In [12]:
# Check number of shards for the confusion_matrix_ln table and print row counts per shard
with head_client.get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM pg_dist_shard WHERE logicalrelid = 'confusion_matrix_ln'::regclass;")
        num_shards = cur.fetchone()['count']
        print(f"Number of shards for 'confusion_matrix_ln' table: {num_shards}")
        rows = cur.fetchall() if False else []
        cur.execute("""
            SELECT shardid FROM pg_dist_shard
            WHERE logicalrelid = 'confusion_matrix_ln'::regclass
            ORDER BY shardid;
        """)
        shard_ids = [row['shardid'] for row in cur.fetchall()]
        empty_count = 0
        for shard_id in shard_ids:
            cur.execute(f"SELECT count(*) FROM public.confusion_matrix_ln_{shard_id};")
            row_count = cur.fetchone()['count']
            print(f"Shard {shard_id}: {row_count} rows")
            if row_count == 0:
                empty_count += 1
        print(f"Empty shards: {empty_count} out of {num_shards}")


Number of shards for 'confusion_matrix_ln' table: 32
Shard 110796: 0 rows
Shard 110797: 0 rows
Shard 110798: 0 rows
Shard 110799: 0 rows
Shard 110800: 0 rows
Shard 110801: 0 rows
Shard 110802: 0 rows
Shard 110803: 0 rows
Shard 110804: 0 rows
Shard 110805: 0 rows
Shard 110806: 0 rows
Shard 110807: 0 rows
Shard 110808: 0 rows
Shard 110809: 0 rows
Shard 110810: 0 rows
Shard 110811: 0 rows
Shard 110812: 0 rows
Shard 110813: 0 rows
Shard 110814: 0 rows
Shard 110815: 0 rows
Shard 110816: 0 rows
Shard 110817: 0 rows
Shard 110818: 0 rows
Shard 110819: 0 rows
Shard 110820: 0 rows
Shard 110821: 0 rows
Shard 110822: 0 rows
Shard 110823: 0 rows
Shard 110824: 0 rows
Shard 110825: 0 rows
Shard 110826: 0 rows
Shard 110827: 0 rows
Empty shards: 32 out of 32


## 6. Verify Dummy Data in Patch Table

Query the patch table to confirm that dummy data has been inserted correctly.

In [13]:
# Query and display dummy patch data
for row in head_client.fetch_patches(limit=10):
    print(row)

{'patch_id': 8, 'patch_uid': 1007, 'label_class_id': 1, 'image_id': 1, 'working_mag': 20.0, 'patch_image': b';\xb5\xc2|\x9a\x1a\x84\xa5w\xc7`\x16B\xc2\xe5L\x05|\xdb})\xbe\xd6\x12\xc44l\x07\xa2\xddD\xce\x16\xce\x0etE`4\xfa\xfc\xb6\xc1\xb9\x9a\x19O\\\x1e\xe5\xc7\xeb5\x01\x11`\xab\x0b|\xa0\xa6S\xd5\x80\x92g\x06\x1b\x04\xc1\xa9P\x87\x0b\x9dTD\xe4>\x18\x0bm\xbd\xe1\x01\xa0\xc0e\xe8\xf8\x9ay\x89\x15\x85M\xef\x17P\x96V\xd1*\xe4\xd0:09{\x13\x89\xf4\xa7\x03\xe8B1\x140\xe2\x16F\x84\xe9\x02bPV'}
{'patch_id': 20, 'patch_uid': 1019, 'label_class_id': 1, 'image_id': 1, 'working_mag': 20.0, 'patch_image': b'`\x14jLz~\xb3SJ\xc6\x9e\x10\xf0W\x1a\xbdy\x15l\xcc\x04{\xa1\xac1f\xb8\xb4\xfb\xfe\xf8\xbd\xb0\xd3\xf8gd\xbd\xa09\xc3\xe3\xff\x9f\xdd\xac\xea\xd6\x97"%,\xcd\xaa\xbf\xd8maG\xc4\x17\xb6\x18\x10\x08dV\x91\x00\x82\xd6\x06\x8b,\x93\xc2\xc6\n\xa4;\x81\xff|\n\xd4\x8b\x17\t\xe3\xc2\r\xd6\xa4\x9a\xd6\xed\x0ciB\xff\xb6\x96\xdae\x97;$$\x0e,\x1e\xa0\n\x92/\x9f\xba\xf4\xa0\xfcx\xdf\xfa\xccz\x16\xb7\x07'}
{'patc

## 7. Implement db_client SDK: Head Node Level

Write Python classes and functions in `db_client.py` to interact with the database at the Citus head node level, including connection management and basic CRUD operations.

In [14]:
# db_client.py will be implemented in the next step.
# Example usage for SDK will be shown after SDK implementation.

In [15]:
# Example: Using db_client SDK (single-node)
# Uses constants.py for all connection parameters
head_client = CitusHeadClient()
print('Patches:', head_client.fetch_patches(limit=3))

# Insert a new patch (dummy data)
# patch_id = head_client.insert_patch(2000, 1, 1, 20.0, b'dummybytes')
# print('Inserted patch_id:', patch_id)

Patches: [{'patch_id': 8, 'patch_uid': 1007, 'label_class_id': 1, 'image_id': 1, 'working_mag': 20.0, 'patch_image': b';\xb5\xc2|\x9a\x1a\x84\xa5w\xc7`\x16B\xc2\xe5L\x05|\xdb})\xbe\xd6\x12\xc44l\x07\xa2\xddD\xce\x16\xce\x0etE`4\xfa\xfc\xb6\xc1\xb9\x9a\x19O\\\x1e\xe5\xc7\xeb5\x01\x11`\xab\x0b|\xa0\xa6S\xd5\x80\x92g\x06\x1b\x04\xc1\xa9P\x87\x0b\x9dTD\xe4>\x18\x0bm\xbd\xe1\x01\xa0\xc0e\xe8\xf8\x9ay\x89\x15\x85M\xef\x17P\x96V\xd1*\xe4\xd0:09{\x13\x89\xf4\xa7\x03\xe8B1\x140\xe2\x16F\x84\xe9\x02bPV'}, {'patch_id': 20, 'patch_uid': 1019, 'label_class_id': 1, 'image_id': 1, 'working_mag': 20.0, 'patch_image': b'`\x14jLz~\xb3SJ\xc6\x9e\x10\xf0W\x1a\xbdy\x15l\xcc\x04{\xa1\xac1f\xb8\xb4\xfb\xfe\xf8\xbd\xb0\xd3\xf8gd\xbd\xa09\xc3\xe3\xff\x9f\xdd\xac\xea\xd6\x97"%,\xcd\xaa\xbf\xd8maG\xc4\x17\xb6\x18\x10\x08dV\x91\x00\x82\xd6\x06\x8b,\x93\xc2\xc6\n\xa4;\x81\xff|\n\xd4\x8b\x17\t\xe3\xc2\r\xd6\xa4\x9a\xd6\xed\x0ciB\xff\xb6\x96\xdae\x97;$$\x0e,\x1e\xa0\n\x92/\x9f\xba\xf4\xa0\xfcx\xdf\xfa\xccz\x16\xb7\x

<!-- Worker node logic omitted for single-node setup -->